In [1]:
# Thêm vào đầu test1.py
import os
print("Current dir:", os.getcwd())
print("CSV exists:", os.path.exists("symbols_list.csv"))

import MetaTrader5 as mt5
print("MT5 init:", mt5.initialize())
print("Account:", mt5.account_info().login if mt5.account_info() else None)

# Kiểm tra 1 symbol
sym = "EURUSD"
print(f"{sym} info:", mt5.symbol_info(sym))
print(f"{sym} selected:", mt5.symbol_select(sym, True))

Current dir: c:\Users\minhh\Downloads\trade-main\trade-main\examples
CSV exists: True
MT5 init: True
Account: 52575885
EURUSD info: SymbolInfo(custom=False, chart_mode=0, select=True, visible=True, session_deals=0, session_buy_orders=0, session_sell_orders=0, volume=0, volumehigh=0, volumelow=0, time=1762416424, digits=5, spread=8, spread_float=True, ticks_bookdepth=10, trade_calc_mode=0, trade_mode=4, start_time=0, expiration_time=0, trade_stops_level=0, trade_freeze_level=0, trade_exemode=2, swap_mode=1, swap_rollover3days=3, margin_hedged_use_leg=False, expiration_mode=15, filling_mode=2, order_mode=127, order_gtc_mode=0, option_mode=0, option_right=0, bid=1.1512, bidhigh=1.1513200000000001, bidlow=1.14765, ask=1.15128, askhigh=1.1514, asklow=1.14919, last=0.0, lasthigh=0.0, lastlow=0.0, volume_real=0.0, volumehigh_real=0.0, volumelow_real=0.0, option_strike=0.0, point=1e-05, trade_tick_value=1.0, trade_tick_value_profit=1.0, trade_tick_value_loss=1.0, trade_tick_size=1e-05, trade_c

In [3]:
# symbols.ipynb
import pandas as pd
import MetaTrader5 as mt5
import os
import json
from datetime import datetime

# === 1. Kiểm tra MT5 ===
if not mt5.initialize():
    raise RuntimeError("Không kết nối được MT5!")

print("MT5 Connected")
print(f"Account: {mt5.account_info().login}")
print(f"Balance: {mt5.account_info().balance}")

# === 2. Đọc CSV ===
csv_path = "symbols_list.csv"  # Điều chỉnh nếu cần
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Không tìm thấy: {csv_path}")

df = pd.read_csv(csv_path)
print(f"Đã load {len(df)} symbols từ CSV")

# === 3. Lọc các cặp có thể trade ===
tradable = []

for _, row in df.iterrows():
    symbol = row['symbol'].strip()
    visible = str(row['is_visible']).lower() == 'true'
    trade_mode = int(row['trade_mode'])

    # Điều kiện cơ bản
    if not (visible and trade_mode == 4):
        continue

    # Chỉ lấy Forex (6 ký tự, không phải kim loại/index)
    if len(symbol) != 6 or any(x in symbol for x in ["XAU", "XAG", "US30", "BTC", "DXY", "USTEC"]):
        continue

    # Kiểm tra trong MT5
    if not mt5.symbol_select(symbol, True):
        continue

    info = mt5.symbol_info(symbol)
    if not info:
        continue

    # Kiểm tra spread hợp lý (tối đa 30 = 3.0 pips)
    if info.spread > 30:
        continue

    # Kiểm tra volume min hợp lý
    if info.volume_min > 1.0:
        continue

    # Tính pip size
    point = info.point
    pip = point * 10 if info.digits >= 4 else point

    tradable.append({
        "symbol": symbol,
        "description": info.description,
        "spread": info.spread,
        "digits": info.digits,
        "point": point,
        "pip": pip,
        "volume_min": info.volume_min,
        "contract_size": info.trade_contract_size,
        "category": info.path.split("\\")[0] if "\\" in info.path else "Unknown"
    })

# === 4. Sắp xếp theo spread (tốt nhất trước) ===
tradable = sorted(tradable, key=lambda x: x["spread"])

# === 5. In kết quả ===
print(f"\nTỔNG CỘNG: {len(tradable)} CẶP CÓ THỂ TRADE THEO MACD")
print("-" * 80)

for i, s in enumerate(tradable[:50], 1):  # Top 50
    print(f"{i:2}. {s['symbol']:6} | Spread: {s['spread']:2} | Pip: {s['pip']:.6f} | VolMin: {s['volume_min']}")

# === 6. Xuất ra file ===
# 6.1. Python list
with open("tradable_forex.py", "w", encoding="utf-8") as f:
    f.write(f"# tradable_forex.py - Cập nhật: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
    f.write("TRADABLE_FOREX = [\n")
    for s in tradable:
        f.write(f"    \"{s['symbol']}\",\n")
    f.write("]\n")
    print(f"\nExported: tradable_forex.py ({len(tradable)} cặp)")

# 6.2. JSON
with open("tradable_forex.json", "w", encoding="utf-8") as f:
    json.dump(tradable, f, indent=2, ensure_ascii=False)
    print("Exported: tradable_forex.json")


print("\nHOÀN TẤT! Dùng `TRADABLE_FOREX` trong bot.py")

MT5 Connected
Account: 52575885
Balance: 7768.0
Đã load 2022 symbols từ CSV

TỔNG CỘNG: 25 CẶP CÓ THỂ TRADE THEO MACD
--------------------------------------------------------------------------------
 1. EURUSD | Spread:  8 | Pip: 0.000100 | VolMin: 0.01
 2. GBPUSD | Spread:  8 | Pip: 0.000100 | VolMin: 0.01
 3. USDCHF | Spread:  8 | Pip: 0.000100 | VolMin: 0.01
 4. USDJPY | Spread:  8 | Pip: 0.001000 | VolMin: 0.01
 5. USDCAD | Spread:  8 | Pip: 0.000100 | VolMin: 0.01
 6. AUDUSD | Spread:  8 | Pip: 0.000100 | VolMin: 0.01
 7. EURGBP | Spread: 10 | Pip: 0.000100 | VolMin: 0.01
 8. EURJPY | Spread: 10 | Pip: 0.001000 | VolMin: 0.01
 9. NZDUSD | Spread: 10 | Pip: 0.000100 | VolMin: 0.01
10. AUDCAD | Spread: 11 | Pip: 0.000100 | VolMin: 0.01
11. AUDCHF | Spread: 11 | Pip: 0.000100 | VolMin: 0.01
12. EURCHF | Spread: 11 | Pip: 0.000100 | VolMin: 0.01
13. NZDCAD | Spread: 11 | Pip: 0.000100 | VolMin: 0.01
14. NZDCHF | Spread: 11 | Pip: 0.000100 | VolMin: 0.01
15. NZDJPY | Spread: 11 | Pip: 